In [1]:
import sympy as sp

In [2]:
sp.init_printing(use_unicode = True)

In [3]:
def escalonar_com_restricoes(matriz):
    M = sp.Matrix(matriz)
    linhas, colunas = M.shape
    restricoes = []

    print("=== MATRIZ INICIAL ===")
    sp.pprint(M)
    print("-" * 50)

    pivo_lin = 0
    for j in range(colunas):
        if pivo_lin >= linhas:
            break

        # 1. Pivotamento (Troca de linhas se o pivô for zero literal)
        if M[pivo_lin, j] == 0:
            for i in range(pivo_lin + 1, linhas):
                if M[i, j] != 0:
                    M.swapr(pivo_lin, i)
                    print(f"\n[TROCA] L{pivo_lin + 1} <-> L{i + 1}")
                    sp.pprint(M)
                    break

        elemento_pivo = M[pivo_lin, j]

        # Se o elemento não for zero literal, mas contiver símbolos
        if elemento_pivo != 0:
            # Registrar restrição se o pivô for simbólico
            if elemento_pivo.free_symbols:
                condicao = sp.Ne(elemento_pivo, 0)
                if condicao not in restricoes:
                    restricoes.append(condicao)

            # 2. Normalização (L = L / pivo)
            if elemento_pivo != 1:
                fator = sp.simplify(1 / elemento_pivo)
                M.row_op(pivo_lin, lambda v, _: sp.simplify(v * fator))
                print(f"\n[NORMALIZAR] L{pivo_lin + 1} = L{pivo_lin + 1} / ({elemento_pivo})")
                print(f"\t-> Condição: {elemento_pivo} != 0")
                sp.pprint(M)

            # 3. Eliminação (Zerar coluna)
            ops = []
            for i in range(linhas):
                if i != pivo_lin and M[i, j] != 0:
                    fator = sp.simplify(-M[i, j])
                    ops.append(f"L{i + 1} = L{i + 1} + ({fator})*L{pivo_lin + 1}")
                    M.row_op(i, lambda v, k: sp.simplify(v + fator * M[pivo_lin, k]))

            if ops:
                print(f"\n[ELIMINAÇÃO] Coluna {j + 1}:")
                for op in ops:
                    print(f"\t{op}")
                sp.pprint(M)

            pivo_lin += 1
            print("-" * 50)

    print("\n" + "=" * 30)
    print("RESULTADO FINAL")
    sp.pprint(M)
    print(f"\nPOSTO (assumindo restrições): {M.rank()}")

    if restricoes:
        print("\nDOMÍNIO DE EXISTÊNCIA (Para este escalonamento ser válido):")
        for res in restricoes:
            sp.pprint(res)

    else:
        print("\nDOMÍNIO: Válido para quaisquer valores reais (sem divisões simbólicas).")
    print("=" * 30)

In [4]:
lamb = sp.Symbol("λ")

matriz = [
    [3, lamb, 1, 2],
    [1, 4, 7, 2],
    [1, 10, 17, 4],
    [4, 1, 3, 3]
]

In [5]:
escalonar_com_restricoes(matriz)

=== MATRIZ INICIAL ===
⎡3  λ   1   2⎤
⎢            ⎥
⎢1  4   7   2⎥
⎢            ⎥
⎢1  10  17  4⎥
⎢            ⎥
⎣4  1   3   3⎦
--------------------------------------------------

[NORMALIZAR] L1 = L1 / (3)
	-> Condição: 3 != 0
⎡   λ           ⎤
⎢1  ─   1/3  2/3⎥
⎢   3           ⎥
⎢               ⎥
⎢1  4    7    2 ⎥
⎢               ⎥
⎢1  10  17    4 ⎥
⎢               ⎥
⎣4  1    3    3 ⎦

[ELIMINAÇÃO] Coluna 1:
	L2 = L2 + (-1)*L1
	L3 = L3 + (-1)*L1
	L4 = L4 + (-4)*L1
⎡      λ               ⎤
⎢1     ─     1/3   2/3 ⎥
⎢      3               ⎥
⎢                      ⎥
⎢        λ             ⎥
⎢0   4 - ─   20/3  4/3 ⎥
⎢        3             ⎥
⎢                      ⎥
⎢        λ             ⎥
⎢0  10 - ─   50/3  10/3⎥
⎢        3             ⎥
⎢                      ⎥
⎢       4⋅λ            ⎥
⎢0  1 - ───  5/3   1/3 ⎥
⎣        3             ⎦
--------------------------------------------------

[NORMALIZAR] L2 = L2 / (4 - λ/3)
	-> Condição: 4 - λ/3 != 0
⎡      λ                   ⎤
⎢1     ─     